In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

df_raw = pd.read_csv('data/proj_machine_readings.csv')
df = df_raw.copy()   

In [2]:
df.head()

,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,failure_type,downtime_risk,maintenance_required
0,2025-01-01 00:00:00,39,NaN,-28.65,79.96,3.73,2.16,1,0,106,Normal,0.0,0
1,2025-01-01 00:01:00,29,68.19,57.28,35.94,3.64,0.69,1,0,320,Normal,0.0,0
2,2025-01-01 00:02:00,15,98.94,50.20,72.06,1.00,2.49,1,1,19,Normal,1.0,1
3,2025-01-01 00:03:00,43,90.91,37.65,30.34,3.15,4.96,1,1,10,Normal,1.0,1
4,2025-01-01 00:04:00,8,72.32,40.69,56.71,2.68,0.63,2,0,65,Vibration Issue,0.0,1


In [3]:
print(df.shape)
df.info()

(100000, 13)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   timestamp                 100000 non-null  str    
 1   machine_id                100000 non-null  int64  
 2   temperature               86000 non-null   float64
 3   vibration                 100000 non-null  float64
 4   humidity                  100000 non-null  float64
 5   pressure                  100000 non-null  float64
 6   energy_consumption        100000 non-null  float64
 7   machine_status            100000 non-null  int64  
 8   anomaly_flag              100000 non-null  int64  
 9   predicted_remaining_life  100000 non-null  int64  
 10  failure_type              100000 non-null  str    
 11  downtime_risk             100000 non-null  float64
 12  maintenance_required      100000 non-null  int64  
dtypes: float64(6), int64(5), str(2)
memory usag

In [4]:
# Insights: initial data
# 100,000 rows, 13 columns.
# Only temperature has nulls: 14,000 missing (14%).
# timestamp is a string -> convert to datetime.
# machine_id is int64 but it is a category label -> convert to category.

In [5]:
# Fix types so the columns are proper
df['timestamp'] = pd.to_datetime(df['timestamp'])        
df['machine_id'] = df['machine_id'].astype('category')  
print(df.dtypes)

timestamp                   datetime64[us]
machine_id                        category
temperature                        float64
vibration                          float64
humidity                           float64
pressure                           float64
energy_consumption                 float64
machine_status                       int64
anomaly_flag                         int64
predicted_remaining_life             int64
failure_type                           str
downtime_risk                      float64
maintenance_required                 int64
dtype: object


In [6]:
# Time check: when does the data start and end, and does any timestamp repeat?
print('Time range:', df['timestamp'].min(), 'to', df['timestamp'].max())   # start and end dates
print('Time span:', df['timestamp'].max() - df['timestamp'].min())         # how long that is
print('Unique timestamps:', df['timestamp'].nunique())                     # compare with number of rows

Time range: 2025-01-01 00:00:00 to 2025-03-11 10:39:00
Time span: 69 days 10:39:00
Unique timestamps: 100000


In [ ]:
# Machine check: how many machines, and does each have a similar number of rows (readings)?
print('Machines:', df['machine_id'].nunique())            
print(df['machine_id'].value_counts().describe())         # rows per machine: min, max, mean

Machines: 50
count      50.000000
mean     2000.000000
std        51.673228
min      1906.000000
25%      1956.250000
50%      2002.500000
75%      2042.500000
max      2092.000000
Name: count, dtype: float64


In [8]:
# how many minutes between one machine's readings?
gaps = df.sort_values('timestamp').groupby('machine_id', observed=True)['timestamp'].diff().dt.total_seconds() / 60
print(gaps.describe())

count    99950.000000
mean        49.973307
std         49.273879
min          1.000000
25%         15.000000
50%         35.000000
75%         69.000000
max        514.000000
Name: timestamp, dtype: float64


In [9]:
# Insights: time span and machines
# Time: data runs from 2025-01-01 00:00 to 2025-03-11 10:39 (69 days).
# Rows: 100,000 rows, each with a different timestamp -> 1 reading per minute is recorded from one of the 50 machines.
#       Each row comes from one of the 50 machines, so a machine does not report every minute.
# Machines: 50 machines with ~2,000 rows each (min 1,906, max 2,092) -> so every machine has about the same number of rows.
# Gaps: each machine is read about every 50 min on average, but unevenly (median 35 min, longest 514 min).

In [10]:
num_cols = ['temperature', 'vibration', 'humidity', 'pressure', 'energy_consumption']
display(df[num_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
temperature,86000.0,75.034410,10.036257,35.55,68.28,75.090,81.78,121.94
vibration,100000.0,43.324232,29.134306,-106.98,37.22,48.635,59.25,113.80
humidity,100000.0,54.995401,14.437960,30.00,42.52,54.980,67.50,80.00
pressure,100000.0,3.000405,1.152399,1.00,2.00,3.010,4.00,5.00
energy_consumption,100000.0,2.747064,1.297865,0.50,1.63,2.740,3.87,5.00


In [11]:
# Insights: value ranges
# vibration has negative values (min -106.98), which is not correct -> investigate later.
# temperature: 14,000 nulls; max 121.94 is far above the mean (75), so check for outliers.
# humidity (30-80), pressure (1-5), energy (0.5-5) look valid.

In [ ]:
# Find the rows where vibration is negative
neg_vib = df[df['vibration'] < 0]

# How many are there?
print('Negative count:', len(neg_vib))

# What percent of all rows is that?  (negative rows / all rows * 100)
print('Percent:', round(len(neg_vib) / len(df) * 100, 2), '%')

# What do the negative vibration values look like?
print(neg_vib['vibration'].describe())

Negative count: 6703
Percent: 6.7 %
count    6703.000000
mean      -49.914617
std        15.246465
min      -106.980000
25%       -60.050000
50%       -50.020000
75%       -40.025000
max        -0.020000
Name: vibration, dtype: float64


In [ ]:
# Insights: negative vibration
# 6,703 rows (6.7%) have negative vibration, which is not correct for mm/s.
# Their values are centred around -50 (mean).
# Ignoring the minus sign, their size is mostly 40-60, which looks like normal vibration.
# Need to check if the vibration sign is wrong or the value itself is wrong.